In [12]:
import os
print(os.listdir("/content"))

['.config', 'sample_data']


In [13]:
import pandas as pd

file_path = "/Traffic_Crashes_-_People_20261111.csv"

df_preview = pd.read_csv(file_path, nrows=5)

print(df_preview["CRASH_DATE"].head())

0    11/10/2026 09:38:00 PM
1    11/10/2026 09:38:00 PM
2    11/10/2026 09:30:00 PM
3    11/10/2026 09:00:00 PM
4    11/10/2026 09:00:00 PM
Name: CRASH_DATE, dtype: object


In [14]:

import pandas as pd

file_path = "/Traffic_Crashes_-_People_20261111.csv"
output_file = "/content/Traffic_Crashes_People_2025_2026.csv"

total_rows = 0

for i, chunk in enumerate(
    pd.read_csv(file_path, chunksize=100000, low_memory=False)
):
    chunk["CRASH_DATE"] = pd.to_datetime(
        chunk["CRASH_DATE"],
        format="%m/%d/%Y %I:%M:%S %p",
        errors="coerce"
    )

    filtered = chunk[
        chunk["CRASH_DATE"].dt.year.isin([2025, 2026])
    ]

    filtered.to_csv(
        output_file,
        mode="w" if i == 0 else "a",
        header=(i == 0),
        index=False
    )

    total_rows += len(filtered)

print("Finished filtering!")
print("Total rows from 2025 and 2026:", total_rows)


Finished filtering!
Total rows from 2025 and 2026: 449848


In [15]:

import pandas as pd

df = pd.read_csv(
    "/content/Traffic_Crashes_People_2025_2026.csv",
    low_memory=False
)

print("Shape:", df.shape)

print("\nMissing values by column:")
print(df.isnull().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)


Shape: (449848, 29)

Missing values by column:
CELL_PHONE_USE           449845
BAC_RESULT VALUE         449405
EMS_RUN_NO               444214
PEDPEDAL_VISIBILITY      438340
PEDPEDAL_LOCATION        438319
PEDPEDAL_ACTION          438317
EMS_AGENCY               418092
HOSPITAL                 397344
SEAT_NO                  361527
DRIVERS_LICENSE_CLASS    240569
DRIVERS_LICENSE_STATE    181951
ZIPCODE                  138340
AGE                      127145
CITY                     121350
STATE                    115515
DRIVER_VISION             89278
BAC_RESULT                89152
DRIVER_ACTION             89082
PHYSICAL_CONDITION        88863
VEHICLE_ID                11770
AIRBAG_DEPLOYED           11317
SEX                        8583
EJECTION                   6398
SAFETY_EQUIPMENT           1109
INJURY_CLASSIFICATION        72
PERSON_ID                     0
PERSON_TYPE                   0
CRASH_RECORD_ID               0
CRASH_DATE                    0
dtype: int64

Duplicate r

In [16]:

print("Duplicate PERSON_ID values:",
      df["PERSON_ID"].duplicated().sum())

print("\nPerson types:")
print(df["PERSON_TYPE"].value_counts(dropna=False))

print("\nInjury classifications:")
print(df["INJURY_CLASSIFICATION"].value_counts(dropna=False))


Duplicate PERSON_ID values: 0

Person types:
PERSON_TYPE
DRIVER                 349778
PASSENGER               88321
PEDESTRIAN               6028
BICYCLE                  5132
NON-MOTOR VEHICLE         531
NON-CONTACT VEHICLE        58
Name: count, dtype: int64

Injury classifications:
INJURY_CLASSIFICATION
NO INDICATION OF INJURY     401848
NONINCAPACITATING INJURY     24750
REPORTED, NOT EVIDENT        19768
INCAPACITATING INJURY         3232
FATAL                          178
NaN                             72
Name: count, dtype: int64


In [17]:

print("Age summary:")
print(df["AGE"].describe())

print("\nNegative ages:",
      (df["AGE"] < 0).sum())

print("\nAges over 110:",
      (df["AGE"] > 110).sum())


Age summary:
count    322703.000000
mean         38.311085
std          17.085040
min           0.000000
25%          26.000000
50%          36.000000
75%          50.000000
max         109.000000
Name: AGE, dtype: float64

Negative ages: 0

Ages over 110: 0


In [18]:
age_missing = pd.crosstab(
    df["PERSON_TYPE"],
    df["AGE"].isna(),
    normalize="index"
) * 100

age_missing.columns = ["Age Present (%)", "Age Missing (%)"]

print(age_missing.round(2))


                     Age Present (%)  Age Missing (%)
PERSON_TYPE                                          
BICYCLE                        83.40            16.60
DRIVER                         73.70            26.30
NON-CONTACT VEHICLE            43.10            56.90
NON-MOTOR VEHICLE              61.21            38.79
PASSENGER                      62.06            37.94
PEDESTRIAN                     91.03             8.97


In [19]:

columns = [
    "SEX",
    "SAFETY_EQUIPMENT",
    "AIRBAG_DEPLOYED",
    "EJECTION",
    "DRIVER_ACTION",
    "PHYSICAL_CONDITION"
]

for col in columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))



--- SEX ---
SEX
M      233669
F      166637
X       40959
NaN      8583
Name: count, dtype: int64

--- SAFETY_EQUIPMENT ---
SAFETY_EQUIPMENT
USAGE UNKNOWN                                 252167
SAFETY BELT USED                              165965
NONE PRESENT                                   17144
HELMET NOT USED                                 4341
SAFETY BELT NOT USED                            2255
CHILD RESTRAINT - FORWARD FACING                1677
BICYCLE HELMET (PEDACYCLIST INVOLVED ONLY)      1649
NaN                                             1109
CHILD RESTRAINT - TYPE UNKNOWN                   938
CHILD RESTRAINT - REAR FACING                    878
DOT COMPLIANT MOTORCYCLE HELMET                  658
BOOSTER SEAT                                     508
SHOULD/LAP BELT USED IMPROPERLY                  163
NOT DOT COMPLIANT MOTORCYCLE HELMET              123
CHILD RESTRAINT NOT USED                         119
WHEELCHAIR                                       100
CHILD REST

In [20]:

for col in ["DRIVER_ACTION", "PHYSICAL_CONDITION"]:
    print(f"\n--- {col} ---")

    result = pd.crosstab(
        df["PERSON_TYPE"],
        df[col].isna(),
        normalize="index"
    ) * 100

    result.columns = ["Present (%)", "Missing (%)"]

    print(result.round(2))



--- DRIVER_ACTION ---
                     Present (%)  Missing (%)
PERSON_TYPE                                  
BICYCLE                    98.09         1.91
DRIVER                    100.00         0.00
NON-CONTACT VEHICLE        94.83         5.17
NON-MOTOR VEHICLE          83.99        16.01
PASSENGER                   0.00       100.00
PEDESTRIAN                 90.46         9.54

--- PHYSICAL_CONDITION ---
                     Present (%)  Missing (%)
PERSON_TYPE                                  
BICYCLE                    98.21         1.79
DRIVER                    100.00         0.00
NON-CONTACT VEHICLE        93.10         6.90
NON-MOTOR VEHICLE          82.67        17.33
PASSENGER                   0.00       100.00
PEDESTRIAN                 94.13         5.87


In [21]:

# Check geographic values
for col in ["CITY", "STATE", "ZIPCODE"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(15))

# Check date range
df["CRASH_DATE"] = pd.to_datetime(
    df["CRASH_DATE"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

print("\nEarliest crash:", df["CRASH_DATE"].min())
print("Latest crash:", df["CRASH_DATE"].max())
print("Invalid dates:", df["CRASH_DATE"].isna().sum())

# Count records by year
print("\nRecords by year:")
print(df["CRASH_DATE"].dt.year.value_counts().sort_index())



--- CITY ---
CITY
CHICAGO         234727
NaN             121350
CICERO            3247
SKOKIE            2292
BERWYN            1759
EVANSTON          1616
UNKNOWN           1558
OAK LAWN          1558
DES PLAINES       1506
CALUMET CITY      1358
CHGO              1235
OAK PARK          1177
ELMWOOD PARK      1054
BURBANK           1050
DOLTON            1038
Name: count, dtype: int64

--- STATE ---
STATE
IL     314702
NaN    115515
IN       5613
XX       2177
WI       1543
MI       1104
FL        979
TX        941
CA        708
OH        678
GA        551
NY        439
MN        431
IA        390
MO        372
Name: count, dtype: int64

--- ZIPCODE ---
ZIPCODE
NaN      138340
60629     11922
60639     10186
60623      8511
60632      8228
60617      8054
60651      8037
60620      8020
60619      6967
60609      6673
60634      6652
60641      6603
60628      6574
60618      6033
60625      5864
Name: count, dtype: int64

Earliest crash: 2025-01-01 00:00:00
Latest crash: 2026-11-10 

In [22]:

# Create a working copy
df_clean = df.copy()

# Standardize city names
df_clean["CITY"] = (
    df_clean["CITY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Standardize known Chicago abbreviation
df_clean["CITY"] = df_clean["CITY"].replace({
    "CHGO": "CHICAGO"
})

# Convert unknown geographic labels to missing
df_clean["CITY"] = df_clean["CITY"].replace({
    "UNKNOWN": pd.NA,
    "": pd.NA
})

# Standardize state values
df_clean["STATE"] = (
    df_clean["STATE"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df_clean["STATE"] = df_clean["STATE"].replace({
    "XX": pd.NA,
    "": pd.NA
})

print("Geographic cleaning complete!")

print("\nTop cities:")
print(df_clean["CITY"].value_counts().head(10))


Geographic cleaning complete!

Top cities:
CITY
CHICAGO         235962
CICERO            3247
SKOKIE            2292
BERWYN            1759
EVANSTON          1616
OAK LAWN          1558
DES PLAINES       1506
CALUMET CITY      1358
OAK PARK          1177
ELMWOOD PARK      1054
Name: count, dtype: Int64


In [23]:

df_clean["ZIPCODE"] = (
    df_clean["ZIPCODE"]
    .astype("string")
    .str.strip()
)

zip_valid = df_clean["ZIPCODE"].str.fullmatch(
    r"\d{5}(-\d{4})?",
    na=False
)

print("Valid ZIP format:", zip_valid.sum())
print("Missing ZIP:", df_clean["ZIPCODE"].isna().sum())

print("\nInvalid ZIP examples:")
print(
    df_clean.loc[
        df_clean["ZIPCODE"].notna() & ~zip_valid,
        "ZIPCODE"
    ].value_counts().head(15)
)


Valid ZIP format: 311232
Missing ZIP: 138340

Invalid ZIP examples:
ZIPCODE
UNK        46
UNKNOWN    25
606         7
6629        3
6           3
6065        3
606932      2
60          2
6062        2
606651      2
CHICAGO     2
6069        2
6029        2
4407        2
6061        2
Name: count, dtype: Int64


In [24]:

# Replace invalid ZIP codes with missing values
df_clean.loc[~zip_valid, "ZIPCODE"] = pd.NA

print("ZIP code cleaning complete!")

print("Missing ZIP codes:",
      df_clean["ZIPCODE"].isna().sum())

print("Remaining invalid ZIP codes:",
      (
          df_clean["ZIPCODE"].notna()
          & ~df_clean["ZIPCODE"].str.fullmatch(
              r"\d{5}(-\d{4})?",
              na=False
          )
      ).sum())

print("Dataset shape:", df_clean.shape)


ZIP code cleaning complete!
Missing ZIP codes: 138616
Remaining invalid ZIP codes: 0
Dataset shape: (449848, 29)


In [25]:

print("FINAL DATA QUALITY CHECK")
print("------------------------")

print("Shape:", df_clean.shape)

print("Duplicate PERSON_ID:",
      df_clean["PERSON_ID"].duplicated().sum())

print("Missing crash dates:",
      df_clean["CRASH_DATE"].isna().sum())

print("Records outside 2025-2026:",
      (~df_clean["CRASH_DATE"].dt.year.isin([2025, 2026])).sum())

print("Negative ages:",
      (df_clean["AGE"] < 0).sum())

print("Ages over 110:",
      (df_clean["AGE"] > 110).sum())

print("\nMissing values by column:")
print(
    df_clean.isna()
    .sum()
    .sort_values(ascending=False)
)


FINAL DATA QUALITY CHECK
------------------------
Shape: (449848, 29)
Duplicate PERSON_ID: 0
Missing crash dates: 0
Records outside 2025-2026: 0
Negative ages: 0
Ages over 110: 0

Missing values by column:
CELL_PHONE_USE           449845
BAC_RESULT VALUE         449405
EMS_RUN_NO               444214
PEDPEDAL_VISIBILITY      438340
PEDPEDAL_LOCATION        438319
PEDPEDAL_ACTION          438317
EMS_AGENCY               418092
HOSPITAL                 397344
SEAT_NO                  361527
DRIVERS_LICENSE_CLASS    240569
DRIVERS_LICENSE_STATE    181951
ZIPCODE                  138616
AGE                      127145
CITY                     122908
STATE                    117692
DRIVER_VISION             89278
BAC_RESULT                89152
DRIVER_ACTION             89082
PHYSICAL_CONDITION        88863
VEHICLE_ID                11770
AIRBAG_DEPLOYED           11317
SEX                        8583
EJECTION                   6398
SAFETY_EQUIPMENT           1109
INJURY_CLASSIFICATION     

In [27]:
# Save the cleaned dataset
output_path = "/content/Traffic_Crashes_People_Cleaned_2025_2026.csv"

df_clean.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")
print("Final shape:", df_clean.shape)


Cleaned dataset saved successfully!
Final shape: (449848, 29)


In [28]:
from google.colab import files

files.download(output_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>